# Computer Lab: Dispersion and Chromaticity — local Python version

This notebook replaces the original Sirepo/Elegant workflow with a fully local first-order matrix model. The helper module `uspas_labs.dispersion_chromaticity` is installed from the course GitHub repo and contains the transfer matrices, matching routines, plotting code, and widget wrappers. The notebook cells expose only the physics knobs students should inspect or modify.

The model is intentionally pedagogical: drifts, thick quadrupoles, sector bends, linear dispersion, finite-difference chromaticity, and tune-footprint plots. It is not a production lattice model.

<style>
.answer {
    color: #b00020;
    border-left: 4px solid #b00020;
    padding-left: 0.8em;
    margin: 0.8em 0;
}
.answer table, .answer th, .answer td {
    color: #b00020;
}
.instructor-note {
    color: #7a3b00;
    border-left: 4px solid #7a3b00;
    padding-left: 0.8em;
    margin: 0.8em 0;
}
.note {
    background-color: #f6f8fa;
    padding: 0.75em;
    border-left: 4px solid #999;
}

.widget-slider, .jupyter-widgets.widget-slider {
    width: 900px;
    max-width: 100%;
}
.widget-slider .noUi-horizontal, .jupyter-widgets.widget-slider .noUi-horizontal {
    height: 12px;
}
.widget-slider .noUi-horizontal .noUi-handle,
.jupyter-widgets.widget-slider .noUi-horizontal .noUi-handle {
    width: 24px;
    height: 24px;
    right: -12px;
    top: -7px;
    border-radius: 50%;
}
.widget-slider .widget-readout, .jupyter-widgets.widget-slider .widget-readout {
    min-width: 70px;
}
</style>


## Learning goals

By the end of this lab you should be able to:

1. Explain how a bend creates horizontal dispersion.
2. Use $\sigma_x^2=\epsilon_x\beta_x+\eta_x^2\sigma_\delta^2$ to estimate beam-size growth from momentum spread.
3. Tune a two-bend section to make an achromat.
4. Distinguish local zero dispersion from globally small dispersion.
5. Estimate momentum acceptance from both aperture and chromatic resonance crossing.
6. Interpret a tune footprint on a low-order resonance diagram.


In [ ]:
HELPER_VERSION = "main"  # Replace with a release tag, e.g. "v2026-lab1", for student releases.
HELPER_REPO = "git+https://github.com/r-hensley/uspas-2026-jupyter.git"

%pip install -q --upgrade xsuite
%pip install -q --upgrade --no-cache-dir "{HELPER_REPO}@{HELPER_VERSION}"

import numpy as np
import pandas as pd
import xtrack as xt
from IPython.display import display

from uspas_labs import dispersion_chromaticity as dch
import plotly.io as pio

try:
    from google.colab import output as colab_output
except ImportError:
    pio.renderers.default = "plotly_mimetype"
else:
    colab_output.enable_custom_widget_manager()
    pio.renderers.default = "colab"

pd.set_option("display.precision", 6)
display(dch.check_environment())


## Model conventions

All distances are in meters. Quadrupole strength $k_1$ is in $\mathrm{m}^{-2}$. Positive $k_1$ focuses horizontally and defocuses vertically. The beam has geometric emittance

$$
\epsilon_x=\epsilon_y=6\ \mathrm{mm\,mrad}=6\times 10^{-6}\ \mathrm{m\,rad}.
$$

The visible lattice cells use modern Xsuite `Environment` syntax to name elements and assemble lines. The dispersion/chromaticity calculations are then run with a local first-order matrix model so this notebook remains independent of unavailable Sirepo/Elegant files.

The horizontal vector used internally is $[x,x',\delta]$, where

$$
\delta = \frac{\Delta p}{p_0}
$$

is the fractional momentum deviation of one particle relative to the reference momentum $p_0$. A beam has a distribution of $\delta$ values, and

$$
\sigma_\delta = \mathrm{rms}(\delta)
$$

is the rms fractional momentum spread of that distribution. For the examples with a centered momentum distribution, $\sigma_\delta=0.001$ means a 0.1% rms momentum spread. Formulas for rms beam size and tune footprint use $\sigma_\delta$; maps or tune shifts for a single off-momentum particle use $\delta$.

Bends include first-order dispersion source terms. The aperture exercise uses an rms-envelope proxy, so the aperture condition is explicitly

$$
n_\sigma\sigma_x = R.
$$

The default is $n_\sigma=1$ to match the simplified style of the original lab. You can change it later.


In [ ]:
emit = dch.GEOMETRIC_EMITTANCE
sigma_delta = 1e-3       # 0.1% rms momentum spread
pipe_radius_m = 0.025    # 2.5 cm
n_sigma_aperture = 1.0   # use 1 for the original simplified rms criterion


# A. Dispersion in a FODO lattice

We start from a 5 m FODO cell. The reference case has no bend. The modified case replaces the 0.5 m bend slot with a 20-degree sector bend. This is the local-code replacement for editing the FODO lattice in Sirepo.


## A0. Reference matched FODO cell with no dipole

Run the baseline matched cell. Record the beam sizes at the two focusing-quadrupole halves (`QFa`, `QFb`) and at `QD`. These are your comparison values before adding dipoles.

The cell below builds the FODO lattice in the same style used in current Xsuite examples: create an `xt.Environment`, define named elements with `env.new(...)`, and assemble an `xt.Line` from a list of component names. The same `QFh` and `D` components are intentionally repeated in the line. The local optics table gives those repeated occurrences names like `QFa`, `D1`, `D2`, and `QFb`.

In [ ]:
env_fodo = xt.Environment()
env_fodo["kq"] = 0.6

env_fodo.new("QFh", xt.Quadrupole, length=0.25, k1="kq")
env_fodo.new("D", xt.Drift, length=2.0)
env_fodo.new("QD", xt.Quadrupole, length=0.5, k1="-kq")

fodo_reference_line = env_fodo.new_line(
    name="fodo_reference",
    components=["QFh", "D", "QD", "D", "QFh"],
)
fodo_reference_line.particle_ref = xt.Particles(p0c=1e9, mass0=xt.ELECTRON_MASS_EV, q0=-1)

fodo_reference, optics_fodo_reference = dch.display_fodo_reference_from_xsuite_line(fodo_reference_line)


In [ ]:
# Reference optics plot and summary tables are produced in the setup cell above.


## A1. Add dipoles to the two FODO drifts

Now insert a 20-degree total bend into the FODO cell. The default model below uses two 10-degree sector bends without edge focusing, so the main new effect is horizontal dispersion. Each 2.0 m drift from the reference FODO is split into a 0.75 m drift, a 0.5 m Xsuite `xt.Bend`, and another 0.75 m drift. The same drift and bend component definitions are reused in the `components` list.

In [ ]:
env_bend = xt.Environment()
env_bend["kq"] = env_fodo["kq"]
env_bend["bend_angle"] = np.deg2rad(20.0)
env_bend["bend_angle_each"] = "0.5 * bend_angle"
env_bend["bend_length"] = 0.5

env_bend.new("QFh", xt.Quadrupole, length=0.25, k1="kq")
env_bend.new("D", xt.Drift, length=0.75)
env_bend.new(
    "BEND",
    xt.Bend,
    length="bend_length",
    angle="bend_angle_each",
    k0="bend_angle_each / bend_length",
)
env_bend.new("QD", xt.Quadrupole, length=0.5, k1="-kq")

# Build the FODO cell with two bend occurrences.
fodo_with_bend_line = env_bend.new_line(
    name="fodo_with_bend",
    components=["QFh", "D", "BEND", "D", "QD", "D", "BEND", "D", "QFh"],
)

# Specify particle type.
fodo_with_bend_line.particle_ref = xt.Particles(p0c=1e9, mass0=xt.ELECTRON_MASS_EV, q0=-1)

fodo_with_bend, optics_fodo_bend = dch.display_fodo_bend_from_xsuite_line(fodo_with_bend_line)


### Optional comparison: add bend edge focusing

The default FODO-with-bend model above uses two sector bends **without** edge focusing. That choice isolates the main new effect of the bends: horizontal dispersion. A rectangular-style bend has tilted end faces, so it also changes the betatron optics even when the beam has no momentum spread.

#### Xsuite bend and edge-angle convention

Xsuite's `xt.Bend` is a sector-bend element. In the Xsuite notation below, `length` is the curved reference-trajectory length $L$, `angle` is the magnet bend angle $\alpha$, and the curvature is $h = 1/\rho = \mathrm{angle} / \mathrm{length}$. The entrance and exit pole-face angles are independent parameters: `edge_entry_angle = e_1` and `edge_exit_angle = e_2`. These angles are specified in **radians**.

![Xsuite sector-bend edge-angle definition](https://xsuite.readthedocs.io/en/latest/_images/sbend_edge_definition.png)

Setting `edge_entry_angle = edge_exit_angle = 0` gives the sector-bend convention: the end faces are aligned with the radial sector boundaries and no pole-face edge kick is applied. Nonzero `e_1` or `e_2` means the end face is tilted relative to that sector face, which gives a thin edge-focusing kick.

In the code below, the slider value is in degrees for readability, then converted with `np.deg2rad(...)` and assigned to both ends of both bends:

```python
fodo_with_edges["B1"].edge_entry_angle = edge_angle
fodo_with_edges["B1"].edge_exit_angle = edge_angle
fodo_with_edges["B2"].edge_entry_angle = edge_angle
fodo_with_edges["B2"].edge_exit_angle = edge_angle
```

For this FODO cell, the **20-degree total bend** is split into two `xt.Bend` occurrences, `B1` and `B2`, each bending by 10 degrees. A symmetric rectangular-bend-like setting is therefore about half of each individual magnet bend angle:

```text
B1: angle = 10 deg, edge_entry_angle = edge_exit_angle = 5 deg
B2: angle = 10 deg, edge_entry_angle = edge_exit_angle = 5 deg
```

If the cell used one single 20-degree bend instead, the analogous rectangular-like setting would be 10 degrees at each end of that one magnet.

Edge focusing is separate from dispersion. It does **not** require an energy spread in the beam. If you track a distribution with $\sigma_\delta=0$, dispersion does not add beam-size growth, but edge focusing can still change $\beta_x$, $\beta_y$, and therefore the rms beam sizes.


In [ ]:
def make_fodo_with_edge_focusing(edge_angle_deg):
    edge_angle = np.deg2rad(edge_angle_deg)

    fodo_with_edges = dch.make_fodo_cell(kq=0.6, with_bend=True, bend_angle_deg=20.0)
    for bend_name in ["B1", "B2"]:
        fodo_with_edges[bend_name].edge_entry_angle = edge_angle
        fodo_with_edges[bend_name].edge_exit_angle = edge_angle

    return fodo_with_edges


dch.interactive_fodo_edge_focusing(
    make_fodo_with_edge_focusing,
    optics_fodo_reference,
    optics_fodo_bend,
)


## Q1. Minimum dispersion and beam-size comparison

Find the minimum horizontal dispersion in the FODO cell with the bend. At what element or element region does it occur: focusing quadrupole, defocusing quadrupole, drift, or bend?

Then compare the no-bend and bend cases. Which beam-size differences come from dispersion, and which differences can occur even at $\sigma_\delta=0$ because the bend changed the optics?


In [ ]:
display(dch.compact_table(dch.dispersion_extrema(optics_fodo_bend)))

display(dch.compact_beam_size_comparison_at_elements(optics_fodo_reference, ["QFa", "QD", "QFb"], sigma_delta=0.0))
display(dch.compact_beam_size_comparison_at_elements(optics_fodo_bend, ["QFa", "QD", "QFb"], sigma_delta=0.0))


**Your Q1 answer**

Minimum $\eta_x$ =  
Location =  
Beam-size comparison with the no-bend FODO =  
Which differences are dispersive? Which are optical/edge-focusing differences?


## Q2. Beam size with 0.1% momentum spread

Assume the beam has rms fractional momentum spread

$$
\sigma_\delta = \mathrm{rms}(\delta) = \mathrm{rms}\!\left(\frac{\Delta p}{p_0}\right)=0.001.
$$

This does not mean every particle has $\delta=0.001$; it means the rms width of the beam's $\delta$ distribution is 0.001. At `QFa`, compute the expected horizontal beam size from

$$
\sigma_x = \sqrt{\epsilon_x\beta_x + (\eta_x\sigma_\delta)^2}.
$$

Compare with the $\sigma_\delta=0$ result. Then do the same comparison for $\sigma_y$.


In [ ]:
sigma_delta = 1e-3

dch.plot_beam_size(optics_fodo_bend, sigma_delta=sigma_delta, title="FODO with bends: effect of 0.1% momentum spread")

display(dch.compact_element_center_table(optics_fodo_bend, ["QFa"], sigma_delta=sigma_delta))
display(dch.compact_beam_size_comparison_at_elements(optics_fodo_bend, ["QFa"], sigma_delta=sigma_delta))


**Your Q2 answer**

| quantity at QFa | $\sigma_\delta=0$ | $\sigma_\delta=0.001$ |
|---|---:|---:|
| $\sigma_x$ | | |
| $\sigma_y$ | | |

Explain why the horizontal and vertical sizes behave differently.

## Try it: bend angle and momentum spread

Move the bend-angle and $\sigma_\delta$ sliders. Watch how $\eta_x$, $\sigma_x$, and $\sigma_y$ respond. This replaces the old Sirepo workflow of editing the lattice and rerunning with a new `Sigma DP` value.


In [ ]:
dch.interactive_fodo_dispersion()


# B. Designing a zero-dispersion insert

The next model is a compact two-bend achromat-like section with two 18-degree bends. With all quadrupoles off, a beam that starts with zero dispersion does not end with zero dispersion. We will use quadrupoles to make $\eta_x$ and $\eta_x'$ return to zero.


## B0. Two-bend cell with all quadrupoles off

Use transport optics, not periodic optics, for this first part. The initial conditions are

$$
\beta_x=\beta_y=10\ \mathrm{m},\qquad \alpha_x=\alpha_y=0,
$$

with zero incoming dispersion. Again, the lattice is assembled with Xsuite `Environment` syntax before being converted to the local first-order model used for the achromat exercise.

In [ ]:
env_dba_off = xt.Environment()
env_dba_off["bend_angle"] = np.deg2rad(18.0)
env_dba_off["bend_length"] = 1.0

env_dba_off.new("D0", xt.Drift, length=0.5)
env_dba_off.new("Q2", xt.Quadrupole, length=0.3, k1=0.0)
env_dba_off.new("D1", xt.Drift, length=0.5)
env_dba_off.new("B1", xt.Bend, length="bend_length", angle="bend_angle", k0="bend_angle / bend_length")
env_dba_off.new("D2", xt.Drift, length=2.3)
env_dba_off.new("Q1", xt.Quadrupole, length=0.3, k1=0.0)
env_dba_off.new("D3", xt.Drift, length=2.3)
env_dba_off.new("B2", xt.Bend, length="bend_length", angle="bend_angle", k0="bend_angle / bend_length")
env_dba_off.new("D4", xt.Drift, length=0.5)
env_dba_off.new("Q3", xt.Quadrupole, length=0.3, k1=0.0)
env_dba_off.new("D5", xt.Drift, length=0.5)

two_bend_off_line = env_dba_off.new_line(
    name="two_bend_off",
    components=["D0", "Q2", "D1", "B1", "D2", "Q1", "D3", "B2", "D4", "Q3", "D5"],
)
two_bend_off_line.particle_ref = xt.Particles(p0c=1e9, mass0=xt.ELECTRON_MASS_EV, q0=-1)

two_bend_off, optics_two_bend_off, eta_end_off, etap_end_off = dch.display_dba_insert_from_xsuite_line(two_bend_off_line)


## Q3. End-of-cell dispersion before correction

Record $\eta_x$ and $\eta_x'$ at the end of the uncorrected cell. Does this cell qualify as achromatic at its exit?


In [ ]:
eta_end, etap_end = dch.endpoint_dispersion(two_bend_off)
pd.DataFrame({"quantity": ["end eta_x [m]", "end eta_x'"], "value": [eta_end, etap_end]})


**Your Q3 answer**

$\eta_x(s_\mathrm{end})$ =  
$\eta_x'(s_\mathrm{end})$ =  
Achromatic at the end? Why or why not?


## B1. Scan Q1 to cancel outgoing dispersion

The central quadrupole Q1 changes the dispersion trajectory between the two bends. First inspect the scan. Then use the optimizer result as a check.


In [ ]:
q1_scan = dch.scan_q1_for_achromat(qmin=0.0, qmax=6.0, n=301)
dch.plot_q1_scan(q1_scan, title="Manual scan: endpoint dispersion versus Q1")
display(dch.compact_q1_scan_table(q1_scan))


In [ ]:
q1_achromat = dch.best_q1_for_achromat(qmin=0.0, qmax=6.0)

display(dch.compact_dba_endpoint_table(q1=q1_achromat, q2=0.0, q3=0.0))
print(f"Best Q1 from local scan/optimizer: {q1_achromat:.6f} 1/m^2")


## Q4. Q1 strength for approximately zero outgoing dispersion

Report the Q1 strength you found, with at least two decimal places.

**Your Q4 answer**

$k_{1,\mathrm{Q1}} =$


## Try it: tune the achromat quadrupole by hand

Move Q1 and watch the final values of $\eta_x$ and $\eta_x'$. The best setting is where both endpoint values are near zero.


In [ ]:
dch.interactive_achromat_q1()


## B2. Add focusing so the cell can be repeated

Canceling outgoing dispersion in an open beamline does not automatically make a stable periodic cell. The default focused DBA-like cell keeps the same Q1 achromat condition and adds two flanking quadrupoles Q2 and Q3. Their placement is outside the zero-dispersion insert, so they can change periodic focusing while preserving the local achromat condition.


In [ ]:
central_q1_cell, focused_dba_cell, optics_focused_dba = dch.display_focused_dba_cell(q1_achromat)


## Q5. Maximum dispersion in the focused DBA-like cell

A dispersion-free insert can still have a dispersion bump elsewhere. Report the maximum $\eta_x$ in the stable focused cell and its location.


In [ ]:
display(dch.compact_table(dch.dispersion_extrema(optics_focused_dba)))
display(dch.compact_element_center_table(optics_focused_dba, ["Q1", "B1", "B2", "Q2", "Q3"], sigma_delta=0.0))


**Your Q5 answer**

Maximum $\eta_x$ =  
Location =  
Why this does not contradict “zero-dispersion insert”:


## Momentum acceptance limited by dispersion

With a finite pipe radius, momentum spread can make the horizontal envelope too large. Here $\sigma_\delta$ is the rms width of the beam's $\delta=\Delta p/p_0$ distribution. The simplified criterion used here is

$$
n_\sigma\sqrt{\epsilon_x\beta_x+\eta_x^2\sigma_\delta^2}=R.
$$

The original lab effectively used the $n_\sigma=1$ rms proxy. You can later change `n_sigma_aperture` to test a stricter engineering convention.


## Q6. Momentum-spread limit from a 2.5 cm pipe radius

Using $R=2.5\ \mathrm{cm}$ and $n_\sigma=1$, find the largest $\sigma_\delta$ that can be transported before the horizontal rms envelope reaches the pipe radius.


In [ ]:
aperture_limits = dch.aperture_limit_table(
    optics_focused_dba,
    pipe_radius_m=pipe_radius_m,
    n_sigma=n_sigma_aperture,
)
display(dch.compact_aperture_limit_table(aperture_limits.head(10)))

dispersion_limit = float(aperture_limits.iloc[0]["max_sigma_delta"])
dispersion_limit_location = str(aperture_limits.iloc[0]["element"])
print(f"Dispersion/aperture limit: sigma_delta = {dispersion_limit:.6f} ({100*dispersion_limit:.3f}%)")
print(f"Limiting element: {dispersion_limit_location}")


**Your Q6 answer**

Largest tolerable $\sigma_\delta$ =  
Equivalent percent momentum spread =  
Limiting element/location =


## Q7. Where does beam loss occur first?

If the momentum spread exceeds your Q6 value, identify where the horizontal envelope first reaches or exceeds the aperture. Is it simply the place with the largest $\eta_x$, or does $\beta_x$ also matter?


In [ ]:
sigma_delta_test = 1.10 * dispersion_limit

dch.plot_beam_size_with_aperture(
    optics_focused_dba,
    sigma_delta=sigma_delta_test,
    pipe_radius_m=pipe_radius_m,
    n_sigma=n_sigma_aperture,
)

display(dch.compact_aperture_summary(optics_focused_dba, pipe_radius_m=pipe_radius_m, n_sigma=n_sigma_aperture))


**Your Q7 answer**

Loss/worst-envelope location =  
Reason =


## Try it: aperture and momentum spread

Use the sliders to compare different momentum spreads and different aperture criteria. In particular, compare $n_\sigma=1$, 3, and 5.


In [ ]:
dch.interactive_aperture(optics_focused_dba)


# C. Chromaticity in a ring

Now repeat the stable DBA-like cell 10 times to form a model ring. A ring has betatron tunes $Q_x$ and $Q_y$. Off-momentum particles see different effective focusing, so their tunes shift with the individual particle momentum offset $\delta=\Delta p/p_0$. The slope $dQ/d\delta$ is chromaticity.


In [ ]:
n_ring_cells = 10
ring_cell = focused_dba_cell
ring = dch.repeat_cell(ring_cell, n_cells=n_ring_cells)

ring_info = dch.ring_summary(ring_cell, n_cells=n_ring_cells)

dch.plot_optics(optics_focused_dba, title="One cell of the repeated DBA-like ring")
display(dch.compact_summary_table(ring_info))


## Q8. Record the x and y tunes

Record $Q_x$ and $Q_y$ to three significant figures.


In [ ]:
Qx, Qy = dch.ring_tunes(ring_cell, n_cells=n_ring_cells)
print(f"Qx = {Qx:.6f}")
print(f"Qy = {Qy:.6f}")


**Your Q8 answer**

$Q_x=$  
$Q_y=$


## Q9. Why are the tunes not necessarily equal?

In the earlier no-bend FODO reference case, the horizontal and vertical phase advances were equal because the lattice was symmetric between the two planes. Inspect the beta functions in the DBA-like cell and explain why equal tunes should not be expected here.


In [ ]:
dch.plot_optics(optics_focused_dba, title="Beta functions and dispersion in one DBA-like cell")
display(dch.compact_optics_summary(optics_focused_dba))


**Your Q9 answer**

Reason the horizontal and vertical tunes need not be equal:


## Q10. Tune spread for 0.1% momentum spread

For one particle, the linearized tune shift is $\Delta Q_x=C_x\delta$ and $\Delta Q_y=C_y\delta$. For a beam with rms momentum spread $\sigma_\delta$, the corresponding one-rms tune spread is

$$
\Delta Q_x=C_x\sigma_\delta,\qquad \Delta Q_y=C_y\sigma_\delta.
$$

For this question use $\sigma_\delta=0.001$.


In [ ]:
Cx, Cy = dch.chromaticity_finite_difference(ring_cell, n_cells=n_ring_cells)
chromatic_table = dch.chromatic_spread_table(Qx, Qy, Cx, Cy, sigma_delta=1e-3)
display(dch.compact_summary_table(chromatic_table))


**Your Q10 answer**

$C_x=$  
$C_y=$  
$\Delta Q_x$ for $\sigma_\delta=0.001$ =  
$\Delta Q_y$ for $\sigma_\delta=0.001$ =


## Resonance diagram and chromatic tune footprint

A resonance line satisfies

$$
mQ_x+nQ_y=p,
$$

where $m$, $n$, and $p$ are integers. The resonance order is $|m|+|n|$. In this exercise, assume resonances of order 3 and below are forbidden, while order 4 and above are tolerable.


In [ ]:
# Start with the 0.1% momentum spread used in Q10.
dch.plot_tune_footprint(Qx, Qy, Cx, Cy, sigma_delta=1e-3, resonance_order=3)


## Try it: move the chromatic tune footprint

Increase $\sigma_\delta$ until the tune-footprint line first crosses a resonance of order 3 or lower. Compare your visual estimate with the table in Q11.


In [ ]:
dch.interactive_tune_footprint(Qx, Qy, Cx, Cy)


## Q11. Momentum acceptance from chromatic resonance crossing

Using resonances through order 3, estimate the largest $\sigma_\delta$ that can be tolerated before the chromatic tune footprint hits a forbidden resonance. Report the value to the nearest 0.1% if using the graph, and compare to the algorithmic value in the table.


In [ ]:
resonance_limits = dch.first_resonance_crossing(Qx, Qy, Cx, Cy, max_order=3)
display(dch.compact_resonance_crossing(resonance_limits.head(10)))

chromaticity_limit = float(resonance_limits.iloc[0]["abs_delta_at_crossing"])
nearest_resonance = str(resonance_limits.iloc[0]["resonance"])
print(f"Chromaticity/resonance limit: sigma_delta = {chromaticity_limit:.6f} ({100*chromaticity_limit:.3f}%)")
print(f"Nearest forbidden resonance: {nearest_resonance}")


**Your Q11 answer**

Largest tolerable $\sigma_\delta$ from the resonance diagram =  
Nearest forbidden resonance =


## Q12. Compare dispersion and chromaticity limits

Compare the aperture/dispersion limit from Q6 with the chromaticity/resonance limit from Q11. The smaller value sets the momentum acceptance in this simplified model.


In [ ]:
comparison = dch.compact_acceptance_comparison(
    dispersion_limit_delta=dispersion_limit,
    chromatic_limit_delta=chromaticity_limit,
)
display(comparison)


**Your Q12 answer**

The stricter momentum-acceptance limit is:  
Reason:


# Optional extension exercises

1. Repeat Q6 with `n_sigma_aperture = 3`. Does the aperture limit become more or less restrictive than the chromaticity limit?
2. Change the FODO bend angle in Section A. How does $\eta_x$ scale with bend angle for small angles?
3. Change Q2 and Q3 in the DBA cell using `dch.interactive_dba_stability()`. Find a stable cell with smaller maximum $\beta_y$.
4. Change the number of ring cells. What happens to total tune and chromaticity?
5. Add edge focusing in the FODO or DBA builders. Which conclusions change, and which remain the same?
